# Implementasi Lanjut: Cuckoo Search - K-Means Clustering

Dataset ini sepenuhnya **dinamis**. Anda bisa mengganti data masukan dari 2D, 3D, bahkan 4D. Metode **Elbow** dan mekanisme **PCA (Principal Component Analysis)** di visualisasi akan menyesuaikan dimensinya secara otomatis!

Tahapan utama:
1. **Prapemrosesan Dinamis & Scaling**: Data dibersihkan dan distandarisasi menggunakan `MinMaxScaler` mengingat perbedaan metrik jarak sangat jauh.
2. **Pencarian Jumlah Cluster Optimal (Elbow Method)**: Mengeksekusi observasi klastering k=1 hingga k=10 untuk mencari letak *Elbow / Patahan* menggunakan pendekatan trigonometri otomatis.
3. **Cuckoo Search & K-Means**: Memanfaatkan centroid Cuckoo pada dataset dimensi terpilih.
4. **Visualisasi Dekomposisi PCA / 2D Asli**: Jika yang dipilih 4D/3D maka fitur dimensional akan direduksi ke dimensi ruang 2D melalui *Principal Component Analysis (PCA)* agar plot penyebaran klasterisasi mudah diobservasi secara visual.

In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from IPython.display import display

# Setup seed untuk reproduktibilitas
np.random.seed(42)

## 1. Pilihan Dimensi & Prapemrosesan Data
Di sel ini Anda dapat **"MENGGANTI-GANTI"** dimensi input klaster. (Lihat komentar '# SILAKAN BERMAIN DENGAN PILIHAN DIMENSI')

In [ ]:
df_raw = pd.read_excel('../data/Data Set UMKM.xlsx', header=1)

# Ambil 5 Kolom Penting untuk diproses menjadi fitur Numerik
kolom_dibutuhkan = ['Nama Usaha', 'Sektor Usaha', 'Omset per-Tahun', 'Usia', 'Laki-laki', 'Perempuan']
df = df_raw[kolom_dibutuhkan].copy()

# Membersihkan nilai string '-' dan menggantinya dengan 0
df[['Usia', 'Laki-laki', 'Perempuan']] = df[['Usia', 'Laki-laki', 'Perempuan']].replace('-', 0).fillna(0)
df['Usia'] = pd.to_numeric(df['Usia'], errors='coerce').fillna(0)
df['Laki-laki'] = pd.to_numeric(df['Laki-laki'], errors='coerce').fillna(0)
df['Perempuan'] = pd.to_numeric(df['Perempuan'], errors='coerce').fillna(0)

# BENTUK FITUR:
df['Total Pekerja Numeric'] = df['Laki-laki'] + df['Perempuan']
df['Usia Numeric'] = df['Usia']

# Hapus missing value ekstrim
df = df[df['Omset per-Tahun'] != '-'].dropna()
df = df.reset_index(drop=True)

# FITUR OMSET (MENGELUARKAN NILAI TENGAH)
def extract_omset(val):
    if 'Kurang dari 10 juta' in val:
        return 5.0
    elif '10 juta s/d 25 juta' in val:
        return 17.5
    elif '25 juta s/d 40 juta' in val:
        return 32.5
    elif '40 juta s/d 55 juta' in val:
        return 47.5
    elif '55 juta s/d 70 juta' in val:
        return 62.5
    elif '70 juta s/d 85 juta' in val:
        return 77.5
    elif '85 juta s/d 100 juta' in val:
        return 92.5
    elif '100 juta s/d 120 juta' in val:
        return 110.0
    elif '120 juta s/d 150 juta' in val:
        return 135.0
    elif 'Lebih dari 150 juta' in val:
        return 150.0
    else:
        return 5.0 # default fallback

df['Omset Numeric'] = df['Omset per-Tahun'].apply(extract_omset)

# FITUR SEKTOR (Label Encoder terurut)
le = LabelEncoder()
df['Sektor Numeric'] = le.fit_transform(df['Sektor Usaha'].astype(str))

# =======================================================
# SILAKAN BERMAIN DENGAN PILIHAN DIMENSI DI BAWAH INI
# Hapus tanda pagar (#) pada ukuran dimensi yang ingin diuji:
# Pstikan hanya satu konfigurasi yang menyala (tidak ada tanda pagar di depannya)
# =======================================================

# PILIHAN 1: KEMBALI KE 2 DIMENSI (2D) KLASIK SEPERTI PERMINTAAN PERTAMA
# fitur_yang_dipakai = ['Sektor Numeric', 'Omset Numeric']

# PILIHAN 2: MENGGUNAKAN 3 DIMENSI (3D)
# fitur_yang_dipakai = ['Sektor Numeric', 'Omset Numeric', 'Usia Numeric']

# PILIHAN 3: MENGGUNAKAN 4 DIMENSI (4D) MULTIVARIATE
fitur_yang_dipakai = ['Sektor Numeric', 'Omset Numeric', 'Usia Numeric', 'Total Pekerja Numeric']

X_raw = df[fitur_yang_dipakai].values

# --- MIN-MAX SCALING ---
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"\n===> MENGGUNAKAN DATA {len(fitur_yang_dipakai)} DIMENSI <===")
print("Kolom yang dipakai:", fitur_yang_dipakai)
print("\nTampilan Sebagian Data Preprocessing:")
display(df[['Nama Usaha'] + fitur_yang_dipakai].head(5))

## 2. Penentuan Clustering Ideal Menggunakan Metode Elbow

In [ ]:
def hitung_optimal_k_elbow(data, max_k=10):
    sse = []
    K = range(1, max_k + 1)
    
    for k in K:
        km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
        km.fit(data)
        sse.append(km.inertia_)  

    # Mencari elbow secara otomatis (jarak terjauh dari garis start-end)
    p1 = np.array([K[0], sse[0]])
    p2 = np.array([K[-1], sse[-1]])
    
    distances = []
    for i in range(len(K)):
        p_point = np.array([K[i], sse[i]])
        dist = np.linalg.norm(np.cross(p2 - p1, p1 - p_point)) / np.linalg.norm(p2 - p1)
        distances.append(dist)
        
    best_k = K[np.argmax(distances)]
    
    plt.figure(figsize=(8, 4))
    plt.plot(K, sse, 'bx-')
    plt.vlines(best_k, plt.ylim()[0], plt.ylim()[1], linestyles='dashed', colors='red', label=f'Optimal K = {best_k}')
    plt.xlabel('Jumlah Klaster (K)')
    plt.ylabel('Sum of Squared Errors (SSE)')
    plt.title('Metode L-Bow Terotomatisasi (Elbow Point)')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    return best_k

print("Memulai Kalkulasi Titik Elbow otomatis...")
optimal_k = hitung_optimal_k_elbow(X_scaled)
print(f"Jumlah Cluster (K) Optimal yang digunakan pada Cuckoo Search: {optimal_k}")

## 3. Algoritma Inti Cuckoo Search (Levy Flight & Abandonment)

In [ ]:
def calculate_distance(p1, p2):
    return np.sqrt(np.sum((p1 - p2) ** 2))

def evaluate_sse(data, centroids):
    sse = 0
    for i in range(len(data)):
        distances = [calculate_distance(data[i], c) for c in centroids]
        min_dist = np.min(distances)
        sse += (min_dist ** 2)
    return sse

def evaluate_fitness(sse):
    return 1.0 / (1.0 + sse)

def levy_flight(Lambda=1.5):
    sigma_u = (math.gamma(1 + Lambda) * math.sin(math.pi * Lambda / 2) / 
               (math.gamma((1 + Lambda) / 2) * Lambda * (2 ** ((Lambda - 1) / 2)))) ** (1 / Lambda)
    sigma_v = 1
    
    u = np.random.normal(0, sigma_u, 1)
    v = np.random.normal(0, sigma_v, 1)
    
    step = u / (abs(v) ** (1 / Lambda))
    return step[0]

def cuckoo_search_kmeans(X, k=3, n_nests=5, max_iter=20, pa=0.25):
    n_features = X.shape[1] 
    n_samples = X.shape[0]
    
    min_bounds = np.min(X, axis=0)
    max_bounds = np.max(X, axis=0)
    
    nests = []
    for _ in range(n_nests):
        nest_centroids = np.random.uniform(min_bounds, max_bounds, (k, n_features))
        nests.append(nest_centroids)
    nests = np.array(nests)
    
    fitness_history = []
    best_nest = None
    best_fitness = -1
    
    for iteration in range(max_iter):
        fitness_values = []
        
        for i in range(n_nests):
            sse = evaluate_sse(X, nests[i])
            fitness = evaluate_fitness(sse)
            fitness_values.append(fitness)
            
            if fitness > best_fitness:
                best_fitness = fitness
                best_nest = np.copy(nests[i])
                
        fitness_values = np.array(fitness_values)
        fitness_history.append((iteration+1, 1.0/best_fitness - 1.0, best_fitness))
        
        for i in range(n_nests):
            step_size = levy_flight() * 0.01
            new_nest = nests[i] + step_size * (nests[i] - best_nest)
            
            new_sse = evaluate_sse(X, new_nest)
            new_fitness = evaluate_fitness(new_sse)
            
            if new_fitness > fitness_values[i]:
                nests[i] = new_nest
                fitness_values[i] = new_fitness
                if new_fitness > best_fitness:
                    best_fitness = new_fitness
                    best_nest = np.copy(new_nest)
                    
        worst_indices = np.argsort(fitness_values)
        num_abandon = int(pa * n_nests)
        
        for i in range(num_abandon):
            idx = worst_indices[i]
            rand_nest_idx = np.random.randint(0, n_nests)
            rand_step = np.random.uniform(0, 1) 
            nests[idx] = nests[idx] + rand_step * (best_nest - nests[rand_nest_idx])
            
    return best_nest, fitness_history


## 4. Eksekusi K-Means menggunakan Parameter Otomatis yang Terpilih

In [ ]:
def final_kmeans(X, initial_centroids):
    centroids = np.copy(initial_centroids)
    clusters = np.zeros(len(X))
    
    while True:
        new_clusters = np.zeros(len(X))
        for i in range(len(X)):
            distances = [calculate_distance(X[i], c) for c in centroids]
            new_clusters[i] = np.argmin(distances)
            
        if np.array_equal(clusters, new_clusters):
            break 
            
        clusters = new_clusters
        for j in range(len(centroids)):
            cluster_points = X[clusters == j]
            if len(cluster_points) > 0:
                centroids[j] = np.mean(cluster_points, axis=0)
                
    return clusters, centroids

print(f"Mengeksekusi Cuckoo Search Algoritma Pada Data {len(fitur_yang_dipakai)} Dimensi...")
best_cuckoo_centroids_scaled, iterations_log = cuckoo_search_kmeans(X_scaled, k=optimal_k, n_nests=10, max_iter=30, pa=0.25)

final_labels, kmeans_centroids_scaled = final_kmeans(X_scaled, best_cuckoo_centroids_scaled)

df['Cluster'] = final_labels
df['Cluster'] = df['Cluster'].apply(lambda x: f"C{int(x)+1}")

print("\n================ H A S I L   A K H I R   C L U S T E R I N G ================")
display(df[['Nama Usaha'] + fitur_yang_dipakai + ['Cluster']].head(15))

## 5. Visualisasi Adaptif Berdasarkan Dimensi

In [ ]:
import matplotlib.cm as cm

plt.figure(figsize=(10, 6))
cmap = cm.get_cmap('Set1', optimal_k)

# CEK DIMENSI, JIKA > 2 MAKA GUNAKAN PCA UNTUK DIKEMBALIKAN KE 2D PADA GRAFIK:
if X_scaled.shape[1] > 2:
    print(f"Merubah data {X_scaled.shape[1]}-Dimensi ke 2-Dimensi menggunakan PCA...")
    pca = PCA(n_components=2)
    X_plot = pca.fit_transform(X_scaled)
    centroids_plot = pca.transform(kmeans_centroids_scaled)
    x_label = 'Principal Component 1'
    y_label = 'Principal Component 2'
    title_grafik = f'Pemetaan Clustering UMKM (Reduksi {len(fitur_yang_dipakai)}D ke 2D PCA)\nK-Optimal = {optimal_k}'
else:
    print("Plotting Data 2 Dimensi Klasik...")
    X_plot = X_scaled
    centroids_plot = kmeans_centroids_scaled
    x_label = fitur_yang_dipakai[0] + ' (Scaled)'
    y_label = fitur_yang_dipakai[1] + ' (Scaled)'
    title_grafik = f'Pemetaan Clustering UMKM (Data 2D Asli)\nK-Optimal = {optimal_k}'

for idx, cluster_label in enumerate(sorted(df['Cluster'].unique())):
    cluster_idx = df['Cluster'] == cluster_label
    plt.scatter(X_plot[cluster_idx, 0], X_plot[cluster_idx, 1], 
                color=cmap(idx), label=f'Cluster {cluster_label}', alpha=0.7, edgecolors='w', s=100)

plt.scatter(centroids_plot[:, 0], centroids_plot[:, 1], c='black', marker='X', s=250, label='Centroids Akhir (K-Means)')

plt.title(title_grafik)
plt.xlabel(x_label)
plt.ylabel(y_label)
plt.legend()
plt.grid(True)
plt.show()